# Subplots

A `Plot` is a single panel. To place several panels in one figure, use one of the
four factories, which all return a `Figure`:

- `on.layout(spec, **panels)` draws the arrangement as a multiline string,
- `on.rows(*panels)` stacks the panels vertically,
- `on.cols(*panels)` places them side by side,
- `on.grid(panels, columns)` wraps a list of panels at a column count.

The names describe the *arguments*, not the container: `on.rows(a, b)` means
"a and b are rows". All of them accept `Plot` and `Figure` panels, so they nest
freely.

A few rules worth knowing:

- `share_x` / `share_y` share the scale domain across the panels of a group. They
  default to `share_x=True, share_y=False` on `rows` and `layout`,
  `share_x=False, share_y=False` on `cols`, and `share_x=True, share_y=True` on
  `grid`.
- **Propagation**: a flag passed explicitly propagates down into nested groups;
  a flag left unset lets each nested group use its own default. This is the most
  likely source of surprise.
- When the x axis is shared in a vertical stack, the inner x axis labels are hidden
  and only drawn on the bottom panel. `labels="all"` draws them everywhere.
- Panel-level `.properties()` wins over figure-level `.properties()`, which only
  fills the values a panel left unset.
- `spacing` sets the gap between panels in pixels, and the layout options below
  control how much room the axes get.
- `width` and `height` are the extents of **one panel**, so a figure of three
  columns is about three times as wide, and a panel spanning two columns is twice
  as wide, gap included.
- `heights` and `widths` size the grid **tracks**, one entry per row or per column:
  `int` values are pixels, `float` values are weights normalised by their sum. An
  axis sized by weights is the one case where `width` or `height` is read as the
  total of that axis rather than as the extent of one panel.

> **Known constraint.** In Vega-Lite 5, `selection_interval(bind="scales")` does not
> reliably propagate across concatenated views, so synchronised pan/zoom across
> panels is not available. Sharing a scale *domain* works, interactive zoom does not.

## Setup

In [1]:
# Import to be able to import python package from src
import sys
sys.path.insert(0, '../src')

In [2]:
import pandas as pd
import ontime as on
from darts.datasets import EnergyDataset

Load and prepare the data.

In [3]:
ts = EnergyDataset().load()

In [4]:
df = ts.pd_dataframe()
df = df.interpolate()
cols = ['generation biomass', 'generation solar', 'generation nuclear']
df = df[cols]

In [5]:
ts = on.TimeSeries.from_dataframe(df)

In [6]:
ts_uni = ts['generation solar'].slice(pd.Timestamp('2015'), pd.Timestamp('2016'))
ts_multi = ts.slice(pd.Timestamp('2015'), pd.Timestamp('2016'))

Let's prepare a few short series to plot.

In [7]:
solar = ts_multi['generation solar'].head(400)
nuclear = ts_multi['generation nuclear'].head(400)
biomass = ts_multi['generation biomass'].head(400)

def line(series, title=None):
    plot = on.Plot(series).add(on.marks.line)
    return plot.properties(title=title) if title is not None else plot

## Drawing the layout as a string

`on.layout` takes the arrangement as a multiline string, one character per grid
cell, and one keyword per character. A character repeated over several cells spans
them, and `.` leaves a cell empty.

```
A A B
C C B
```

reads as "two wide panels stacked on the left, one tall panel on the right". Only
`A-Z`, `a-z`, `0-9` and `.` are allowed, whitespace between the cells is free, and
the grid must be rectangular.

The `width` and `height` below size one grid cell, so `A` and `C` come out twice as
wide as `B`, and `B` twice as tall as them.

In [8]:
on.layout(
    "A A B\n"
    "C C B",
    A=line(solar, title='solar'),
    B=line(biomass, title='biomass'),
    C=line(nuclear, title='nuclear'),
)\
    .properties(width=300, height=150)\
    .show()

alt.HConcatChart(...)

A full width banner over two panels, with the rows sized by weight: the banner
takes 70% of the height of the figure and the two panels below share the rest. The
banner spans the two columns, so it is twice as wide as they are.

In [9]:
on.layout(
    "T T\n"
    "A B",
    T=line(solar, title='solar'),
    A=line(nuclear, title='nuclear'),
    B=line(biomass, title='biomass'),
    heights=[0.7, 0.3],
)\
    .properties(width=400, height=400)\
    .show()

alt.VConcatChart(...)

An empty cell keeps the grid rectangular where a panel is missing, here a tall
panel on the left and a single panel in the bottom right corner.

In [10]:
on.layout(
    "A .\n"
    "A B",
    A=line(solar, title='solar'),
    B=line(nuclear, title='nuclear'),
)\
    .properties(width=350, height=150)\
    .show()

alt.HConcatChart(...)

Tracks can also be given in pixels, one entry per column and per row. A panel
spanning two tracks takes their total, the gap between them included.

In [11]:
on.layout(
    "A A B\n"
    "C C B",
    A=line(solar),
    B=line(biomass),
    C=line(nuclear),
    widths=[300, 300, 200],
    heights=[240, 60],
)\
    .show()

alt.HConcatChart(...)

## Sharing a scale between named panels

Booleans share a scale across a whole group. To share it between panels that are
not a group of their own, name them: `share_y="AC"` pins the union of their data
domains on both, and a list of strings makes several independent groups, as in
`share_y=["AC", "BD"]`.

In [12]:
on.layout(
    "A B\n"
    "C .",
    A=line(solar, title='solar'),
    B=line(biomass, title='biomass'),
    C=line(nuclear, title='nuclear'),
    share_y="AC",
)\
    .properties(width=380, height=180)\
    .show()

alt.VConcatChart(...)

## Stacked signals with a shared time axis

The x axis is shared, so its labels are only drawn on the bottom panel.

In [13]:
on.rows(line(solar), line(nuclear), line(biomass))\
    .properties(width=800, height=140)\
    .show()

alt.VConcatChart(...)

## Unequal heights in pixels

A tall main panel with a thin heatmap strip below it.

In [14]:
main = on.Plot(solar).add(on.marks.line)
strip = on.Plot(solar).add(on.marks.heatmap)

on.rows(main, strip, heights=[240, 40])\
    .properties(width=800)\
    .show()

alt.VConcatChart(...)

## Fractional heights

`float` heights are weights: they are normalised by their own sum and share the
height of the figure, minus the gaps between the panels.

In [15]:
forecast = on.Plot(nuclear).add(on.marks.line)
residuals = on.Plot(nuclear.diff()).add(on.marks.line)

on.rows(forecast, residuals, heights=[0.72, 0.28])\
    .properties(width=700, height=340)\
    .show()

alt.VConcatChart(...)

## Side by side comparison

A comparison inverts both defaults: the panels share the y scale so the magnitudes
are comparable, but keep their own x scale because they cover different periods.

In [16]:
ts_2015 = ts['generation solar'].slice(pd.Timestamp('2015-06-01'), pd.Timestamp('2015-06-08'))
ts_2016 = ts['generation solar'].slice(pd.Timestamp('2016-06-01'), pd.Timestamp('2016-06-08'))

on.cols(line(ts_2015, title='2015'), line(ts_2016, title='2016'), share_y=True, share_x=False)\
    .properties(width=350, height=200)\
    .show()

alt.HConcatChart(...)

## Small multiples

`on.grid` wraps a list of panels at a column count, and shares both scales so the
panels are comparable. A last row that is not full is padded with empty cells.

In [17]:
panels = [line(ts_multi[c].head(400), title=c) for c in ts_multi.components]

on.grid(panels, columns=2)\
    .properties(width=350, height=140)\
    .show()

alt.VConcatChart(...)

## Nesting

Groups nest freely. Here the inner `rows` inherits nothing from the outer `cols`,
so it uses its own default `share_x=True`, while the outer `cols` keeps
`share_x=False`.

In [18]:
profile = on.Plot(biomass).add(on.marks.line)

on.cols(on.rows(main, strip, heights=[240, 50]), profile, widths=[620, 180])\
    .properties(height=290)\
    .show()

alt.HConcatChart(...)

## Adjusting the layout

Four options of `Figure.properties()` control how the panels are placed.

- `spacing` is the gap between panels in pixels (default `4`). The gap is measured
  between the *full* bounds of the panels, axes and titles included, so panels never
  overrun each other.
- `bounds` selects how a panel is measured: `"full"` (default) accounts for the axes
  and titles, `"flush"` only accounts for the plotting areas. `"flush"` packs the
  panels tightly but lets axis labels and titles overlap their neighbours, so it is
  only useful when the inner axes are hidden.
- `hide_shared_axes` (default `True`) drops the redundant inner axes of a shared
  scale: the x axis is kept on the bottom row only, and the y axis on the first
  column only. Set it to `False` to draw every axis.
- `axis_extent` (default `40`) is the minimum width reserved for the y axis of every
  panel, which keeps the y axis titles aligned across panels. Use `0` to let each
  panel size its own axis.


In [19]:
on.rows(line(solar), line(nuclear), line(biomass))\
    .properties(width=800, height=140, spacing=24, axis_extent=60)\
    .show()

alt.VConcatChart(...)

The layout of a figure is available as its canonical layout string, which is handy
to check what was built without touching the data. It is what `repr` shows, and it
can be fed back to `on.layout`.

In [20]:
figure = on.rows(line(solar, title='solar'), line(nuclear, title='nuclear'), heights=[0.5, 0.5])
print(figure.to_string())
print(on.layout(figure.to_string(), A=line(solar), B=line(nuclear)).to_string())

A
B
A
B
